# 1. Compile C code and launch front simulations

This notebook is only for producing data. It compiles `abm_front_propagation_glued.c`, reads the `[run]` blocks from the parameter file, and launches the selected runs in parallel.

Expected outputs per run:

- `<param_base>_run_XXX.dat`: sparse particle snapshots.
- `front_<param_base>_run_XXX.dat`: front observables.
- `rho_<param_base>_run_XXX.dat`: density profiles, only if `rho_profile_every_front > 0`.
- `<param_base>_run_XXX.log`: terminal output.

For density validation, add for example `rho_profile_every_front = 5` or `10` to each run block. This means one density profile every 5 or 10 front measurements.

In [ ]:
from pathlib import Path

from front_runner import (
    compile_c_code,
    expected_output_files,
    read_run_blocks,
    run_front_simulations,
    select_runs,
)

project_folder = Path(".").resolve()

c_file = project_folder / "abm_front_propagation_glued.c"
exe_file = project_folder / "abm_front_propagation_glued.exe"
param_file = project_folder / "params_brownian.txt"

param_base = param_file.stem
data_folder = project_folder / "data" / param_base
data_folder.mkdir(parents=True, exist_ok=True)

max_parallel = 12
selected_run_ids = None  # None runs all [run] blocks. Example: [0, 1, 2]
compile_first = True
run_simulations = True
skip_existing_outputs = False

print("Project folder:", project_folder)
print("C file        :", c_file)
print("Executable    :", exe_file)
print("Parameter file:", param_file)
print("Output folder :", data_folder)

## 1. Compile

In [ ]:
if compile_first:
    compile_c_code(c_file, exe_file)
else:
    print("Skipping compilation.")

## 2. Read runs from the parameter file

In [ ]:
runs = read_run_blocks(param_file)
selected_runs = select_runs(runs, selected_run_ids)

print(f"Found {len(runs)} runs in the parameter file.")
print("Selected runs:", [r["run_id"] for r in selected_runs])

## 3. Launch selected runs

In [ ]:
if not run_simulations:
    print("run_simulations = False")
else:
    run_results = run_front_simulations(
        exe_file=exe_file,
        param_file=param_file,
        max_parallel=max_parallel,
        selected_run_ids=selected_run_ids,
        data_folder=data_folder,
        skip_existing=skip_existing_outputs,
        required_output="front",
    )
    run_results

## 4. Check generated files

In [ ]:
for run in selected_runs:
    run_id = int(run["run_id"])
    files = expected_output_files(data_folder, param_base, run_id)
    print(f"run_id = {run_id}")
    for label in ["snapshot", "front", "rho", "log"]:
        path = files[label]
        status = "OK" if path.exists() else "missing"
        print(f"  {status:7s} {label:8s} {path}")